# Gridworld: from logits to a sampled episode

A fixed policy turns the finite-horizon MDP in `gridworld/model.py` into trajectories.
We will inspect action probabilities, draw one complete episode, and separate
policy sampling, environmental slip, and reward noise. There is no training here.

Install `rl_math/requirements.txt` into the notebook's Python environment.
NumPy handles probability vectors, random draws, and vector calculations. Run
the cells in order from the repository root, `rl_math`, or this notebook's directory.


In [1]:
from pathlib import Path
import numpy as np
import platform
import sys

project = next(p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "rl_math")
               if (p / "gridworld" / "model.py").is_file())
sys.path.insert(0, str(project))
from gridworld.model import Action, Gridworld
from common.returns import discounted_returns
from common.policy import Policy
from gridworld.policy import TabularSoftmaxPolicy
from gridworld.sampling import sample_episode

print("Python", platform.python_version(), "NumPy", np.__version__)


Python 3.10.12 NumPy 2.2.6


## The policy interface

`common/policy.py` defines `Policy[Observation]`, an abstract interface for
discrete-action policies. Its single method, `probabilities(observation)`,
returns a normalized NumPy vector in the environment's action order.
The observation type can be a gridworld state or another representation,
such as a vector. `gridworld/policy.py` supplies
`TabularSoftmaxPolicy(Policy[State])`, the gridworld implementation. A rule-based policy can implement the same
method without having logits at all.

`gridworld.sampling.sample_episode` accepts `Policy[State]` and owns the random draws. It
does not inspect the concrete policy's parameters. This keeps environment
sampling separate from how the policy represents action probabilities
and from the algorithm that may later update its parameters.


## One row of a tabular policy

For the full state $s=(\mathrm{cell},\mathrm{remaining})$, assign four logits
$\theta_{s,a}$ in **north, east, south, west** order:

$$\pi_\theta(a\mid s)=\frac{\exp(\theta_{s,a})}{\sum_b\exp(\theta_{s,b})}.$$

The row $(\log 1,\log 2,\log 3,\log 4)$ gives probabilities
$(0.1,0.2,0.3,0.4)$. `probabilities` subtracts the largest logit before
exponentiating; this leaves the ratios unchanged and avoids overflow.


In [2]:
world = Gridworld()  # Default 5x5 map; horizon=12, slip=0.2.
example = TabularSoftmaxPolicy({world.start_state: np.log([1, 2, 3, 4])})
for action, probability in zip(Action, example.probabilities(world.start_state)):
    print(f"{action.name:5s}: {probability:.1f}")
assert np.isclose(example.probabilities(world.start_state).sum(), 1)


NORTH: 0.1
EAST : 0.2
SOUTH: 0.3
WEST : 0.4


## A fixed preference for the longer safe route

The bottom row is `SHHHG`. Going north, east four times, then south avoids
the hazards when all moves succeed. Give the preferred action logit $\log 7$
and every other action logit $0$: the policy samples its preference with
probability $0.7$ and each other action with probability $0.1$.

This is a manually specified policy. Every nonterminal cell/time pair has its
own row, although we use the same preference at a cell for each remaining time.
The policy may choose another action, and slip may change where that action
lands. Even the preferred action does not guarantee a safe trajectory.

A **tabular** policy stores separate parameters for each state rather than
sharing them through a neural network. It can learn when an optimizer
updates those logits. Here they are chosen by hand and held fixed:
collecting episodes only generates data. The gridworld experiments first
hold the policy fixed to compare gradient estimators with an exact reference;
the later CartPole experiments add learning updates.


In [3]:
logits = {}
for state in world.states():
    if world.is_terminal(state):
        continue  # No actions after the goal or the task horizon.
    row, column = state.cell
    preferred = Action.NORTH if row == 4 else Action.EAST if column < 4 else Action.SOUTH
    logits[state] = np.zeros(len(Action))
    logits[state][tuple(Action).index(preferred)] = np.log(7)
policy = TabularSoftmaxPolicy(logits)
print("Start probabilities:", np.round(policy.probabilities(world.start_state), 2))


Start probabilities: [0.7 0.1 0.1 0.1]


## Draw and inspect a complete episode

At each time $t$, `sample_episode` makes three draws:

$$a_t\sim\pi_\theta(\cdot\mid s_t),\qquad
s_{t+1}\sim P(\cdot\mid s_t,a_t),\qquad
r_{t+1}=\bar r(s_t,a_t,s_{t+1})+\epsilon_{t+1},$$

where $\epsilon_{t+1}\sim\mathcal N(0,\sigma^2)$ independently at each step.
Here $\bar r$ is the selected transition's conditional expected reward;
`world.expected_reward(s, a)` additionally averages over possible next states.

Use seed **7**, slip **0.2**, horizon **12**, and reward-noise standard deviation
**2.0** for this illustration. NumPy `SeedSequence(seed).spawn(3)` creates separate
streams for actions, transitions, and reward noise; each uses `default_rng`.
The sampler leaves global random state alone. Repeating a seed replays an episode.
Use distinct nonnegative integer seeds for new samples, and keep the Python/NumPy
environment and all settings fixed when reproducing a run.
See [NumPy's stream-spawning explanation](https://numpy.org/doc/stable/reference/random/parallel.html).


In [4]:
seed, noise_std, gamma = 7, 2.0, 0.9
episode = sample_episode(world, policy, seed=seed, reward_noise_std=noise_std)
assert episode == sample_episode(world, policy, seed=seed, reward_noise_std=noise_std)
print(f"seed={episode.seed}, sigma={episode.reward_noise_std}, horizon={world.horizon}, slip={world.slip}")
print(" t   cell  left action  reward -> next cell left   end")
for step in episode.steps:
    print(f"{step.t:2d} {str(step.state.cell):>7} {step.state.remaining:4d} {step.action.name:5s} "
          f"{step.reward:7.3f} -> {str(step.next_state.cell):>7} "
          f"{step.next_state.remaining:4d} {str(step.terminated):>5}")
print("Ending:", episode.end_reason)
print(f"G_0 (gamma={gamma}) = {discounted_returns([s.reward for s in episode.steps], gamma)[0]:.3f}")


seed=7, sigma=2.0, horizon=12, slip=0.2
 t   cell  left action  reward -> next cell left   end
 0  (4, 0)   12 EAST   -5.921 ->  (4, 1)   11 False
 1  (4, 1)   11 NORTH   1.216 ->  (3, 1)   10 False
 2  (3, 1)   10 EAST   -2.025 ->  (3, 2)    9 False
 3  (3, 2)    9 SOUTH  -4.254 ->  (4, 2)    8 False
 4  (4, 2)    8 EAST   -5.756 ->  (4, 3)    7 False
 5  (4, 3)    7 NORTH   1.383 ->  (3, 3)    6 False
 6  (3, 3)    6 NORTH  -1.627 ->  (2, 3)    5 False
 7  (2, 3)    5 EAST   -2.542 ->  (3, 3)    4 False
 8  (3, 3)    4 NORTH  -0.418 ->  (2, 3)    3 False
 9  (2, 3)    3 EAST   -0.367 ->  (2, 4)    2 False
10  (2, 4)    2 SOUTH  -2.187 ->  (2, 4)    1 False
11  (2, 4)    1 SOUTH  -1.517 ->  (3, 4)    0  True
Ending: horizon
G_0 (gamma=0.9) = -16.169


In [5]:
# Mark all visited cells; the table above gives their order and revisits.
visited = {step.state.cell for step in episode.steps} | {episode.steps[-1].next_state.cell}
for row, line in enumerate(world.layout):
    print(" ".join(symbol + ("*" if (row, column) in visited else " ")
                   for column, symbol in enumerate(line)))
print("* visited; S start, G goal, H hazard, # wall")


.  .  .  .  . 
.  .  #  .  . 
.  .  #  .* .*
.  .* .* .* .*
S* H* H* H* G 
* visited; S start, G goal, H hazard, # wall


In the NumPy seed-7 run, the policy samples EAST at t=0 even though NORTH
has higher probability; this sends it onto a hazard. At t=7 it selects
EAST from (2, 3) but moves SOUTH to (3, 3): that is environmental slip.
The episode ends at the horizon before reaching the goal.

The recorded `t` starts at zero for each episode; it is the exponent in the
outer $\gamma^t$ policy-gradient weight introduced in `common/returns.py`. Every
actual move decreases remaining time by one, including a move blocked by a wall.

The final transition is marked `terminated=True`: reaching the goal or
exhausting this task's horizon gives zero future value. There is no absorbing
tail, rollout cutoff, or external time-limit truncation in this sampler.
Reaching the goal on the last allowed move still pays the goal reward.


## Why noise changes variance, not the expected reward

The logits change action probabilities and thus which states are visited.
The environment supplies slip and reward noise. With this fixed state-only
policy, reward noise does not influence future actions or transitions.

Independence and zero mean give

$$\mathbb E[r_{t+1}\mid s_t,a_t,s_{t+1}]=\bar r(s_t,a_t,s_{t+1}),$$

so averaging over next states also preserves `world.expected_reward(s, a)`.
The added conditional reward variance is $\sigma^2$. Given a state/action path
of length $T$, the noise contribution to its discounted return has mean zero
and variance

$$\operatorname{Var}\!\left[\sum_{t=0}^{T-1}\gamma^t\epsilon_{t+1}\;\middle|\;
\mathrm{path}\right]=\sigma^2\sum_{t=0}^{T-1}\gamma^{2t}.$$

Unconditionally, expected return is also unchanged; the added return variance
averages this expression over paths. A particular noisy episode can have a
higher or lower return. It need not average to zero within that episode.

Changing only the noise scale keeps the same sampled path because its random
stream is separate. Compare the clean and noisy rewards on that path below.


In [6]:
clean = sample_episode(world, policy, seed=seed)
assert [(s.state, s.action, s.next_state) for s in clean.steps] == [
    (s.state, s.action, s.next_state) for s in episode.steps]
print(" t model reward   noise observed reward")
for base, noisy in zip(clean.steps, episode.steps):
    print(f"{base.t:2d} {base.reward:12.3f} {noisy.reward - base.reward:7.3f} {noisy.reward:15.3f}")
times = np.arange(len(episode.steps))
variance = noise_std**2 * np.sum(gamma**(2 * times))
print(f"Added return variance conditional on this path: {variance:.3f}")


 t model reward   noise observed reward
 0       -6.000   0.079          -5.921
 1       -1.000   2.216           1.216
 2       -1.000  -1.025          -2.025
 3       -6.000   1.746          -4.254
 4       -6.000   0.244          -5.756
 5       -1.000   2.383           1.383
 6       -1.000  -0.627          -1.627
 7       -1.000  -1.542          -2.542
 8       -1.000   0.582          -0.418
 9       -1.000   0.633          -0.367
10       -1.000  -1.187          -2.187
11       -1.000  -0.517          -1.517
Added return variance conditional on this path: 19.373
